# Credit Card Fraud Detection
### (1) Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score
)

from imblearn.over_sampling import SMOTE

### (2) Load Dataset

In [ ]:
df = pd.read_csv("creditcard.csv")

print("Shape:", df.shape)
print("\nClass Distribution:\n", df['Class'].value_counts())

### (3) Exploratory Data Analysis (EDA)
#### (a) Class Distribution

In [ ]:
df['Class'].value_counts().plot(kind='bar')
plt.title("Class Distribution")
plt.xticks([0,1], ["Non-Fraud", "Fraud"], rotation=0)
plt.show()

#### (b) Amount Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df[df['Class']==0]['Amount'], bins=50, label='Non-Fraud', stat='density')
sns.histplot(df[df['Class']==1]['Amount'], bins=50, label='Fraud', stat='density')
plt.legend()
plt.title("Transaction Amount Distribution")
plt.show()

#### (c) Time Distribution

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df['Time'], bins=50)
plt.title("Transaction Time Distribution")
plt.show()

#### (d) Statistical Summary

In [ ]:
print(df.groupby('Class')['Amount'].describe())

### (4) Feature Scaling

In [ ]:
scaler = StandardScaler()

df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

df = df.drop(['Amount', 'Time'], axis=1)

### (5) Train-Test Split

In [ ]:
X = df.drop('Class', axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Train fraud ratio:", y_train.mean())
print("Test fraud ratio:", y_test.mean())

### (6) Baseline Model (Logistic Regression)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("\nBaseline Model Evaluation:")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))
print("PR-AUC:", average_precision_score(y_test, y_proba))

### (7) Threshold Tuning

In [ ]:
print("\nThreshold Tuning Results:")

thresholds = np.arange(0.1, 0.9, 0.1)

for t in thresholds:
    y_pred_thresh = (y_proba >= t).astype(int)
    
    precision = precision_score(y_test, y_pred_thresh)
    recall = recall_score(y_test, y_pred_thresh)
    
    print(f"Threshold: {t:.1f} | Precision: {precision:.3f} | Recall: {recall:.3f}")

#### Final Threshold Selection

In [ ]:
threshold = 0.2
y_pred_final = (y_proba >= threshold).astype(int)

print("\nFinal Model (Threshold = 0.2):")
print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

### (8) Class Weight Approach

In [ ]:
model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
model_balanced.fit(X_train, y_train)

y_proba_bal = model_balanced.predict_proba(X_test)[:, 1]
y_pred_bal = (y_proba_bal >= 0.5).astype(int)

print("\nClass Weight Model:")
print(confusion_matrix(y_test, y_pred_bal))
print(classification_report(y_test, y_pred_bal))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_bal))
print("PR-AUC:", average_precision_score(y_test, y_proba_bal))

### (9) SMOTE Approach

In [ ]:
smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\nBefore SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_sm.value_counts())

model_smote = LogisticRegression(max_iter=1000)
model_smote.fit(X_train_sm, y_train_sm)

y_proba_sm = model_smote.predict_proba(X_test)[:, 1]
y_pred_sm = (y_proba_sm >= 0.5).astype(int)

print("\nSMOTE Model:")
print(confusion_matrix(y_test, y_pred_sm))
print(classification_report(y_test, y_pred_sm))
print("ROC-AUC:", roc_auc_score(y_test, y_proba_sm))
print("PR-AUC:", average_precision_score(y_test, y_proba_sm))